# CE541E08 — Unit 3 · Day 27 — Analysis of Arrays: AMS Statistics, Flood Frequency and FDC

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 3 — The NumPy Library |
| **Session** | Day 27 of 45 |
| **CO** | CO3, CO4 |
| **Topics** | AMS descriptive statistics · Weibull flood frequency · np.interp · Flow Duration Curve |

---
> This is the capstone session for Unit 3. All four code blocks build one complete flood hydrology analysis workflow.
> Read the explanation before each block. Check the expected output. Run and verify. Then try the challenge.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 27"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — The Complete Flood Analysis Workflow

Day 27 brings together everything from Unit 3 into a complete flood hydrology analysis:

**Part A — AMS Statistics:** Descriptive statistics of the Annual Maximum Series — the foundation of any flood frequency study.

**Part B — Weibull Flood Frequency Table:** Ranking and plotting position to estimate the probability and return period of each observed peak.

**Part C — Design Flood Estimation:** Using `np.interp` to estimate the T-year design flood (T=2, 5, 10, 25, 50 years) from the Weibull table.

**Part D — Flow Duration Curve:** Building the FDC from a daily record to characterise the river's flow regime.

This is the workflow a junior hydrologist would follow when tasked with "estimate the 100-year flood for this dam spillway design." 

---
## Code Block 1 — Part A: AMS Descriptive Statistics

### What this code does

We compute the full set of descriptive statistics for a 25-year Annual Maximum Series (AMS). This includes standard statistics plus the **Coefficient of Variation (CV)** and **skewness** — both are standard inputs to flood frequency distributions like the Gumbel and Log-Pearson Type III.

### Why each step is taken

**`AMS.std() / AMS.mean()` — Coefficient of Variation (CV):**
CV normalises the standard deviation by the mean, giving a dimensionless measure of relative variability. A CV of 0.5 means the typical year-to-year variation is 50% of the mean flood. Rivers with high CV are flashy and difficult to design for.

**Skewness — `(((AMS-mean)/std)**3).mean()`:**
Skewness measures the asymmetry of the distribution. A positive skew means the distribution has a long right tail — a few very large floods. Most flood datasets have positive skew, which is why the Log-Pearson III distribution (which handles skew) is the US standard for flood frequency analysis.

**`np.median(AMS)` vs `AMS.mean()`:**
If the median is significantly lower than the mean, the distribution is right-skewed. This is typical for flood data — most years are moderate, but occasionally there is an extreme event.

### Algorithm

```
1. Load 25-year AMS (one peak per year)

2. Basic statistics:
   n    = AMS.size
   mean = AMS.mean()
   std  = AMS.std()
   CV   = std / mean
   min  = AMS.min()
   max  = AMS.max()

3. Robust statistics:
   median = np.median(AMS)
   Q25    = np.percentile(AMS, 25)
   Q75    = np.percentile(AMS, 75)

4. Skewness:
   z    = (AMS - mean) / std   → standardised values
   skew = (z**3).mean()        → third moment
```

### Expected output

```
==================================================
PART A — AMS STATISTICS
--------------------------------------------------
Record length :  25 years
Mean          : 3024.0 m3/s
Std deviation : 1114.7 m3/s
CV            : 0.369
Min           : 1234.0
Max           : 5234.0
Median        : 2890.0
Q25           : 2123.0
Q75           : 3890.0
Skewness      : 0.101
```

In [ ]:
import numpy as np

# 25-year Annual Maximum Series (m³/s) — Cauvery at KRS
AMS = np.array([1234, 2456, 1890, 3456, 2234, 4567, 3123, 1567, 2890, 5234,
                3456, 2123, 4567, 2890, 1678, 3234, 4890, 2345, 3678, 1456,
                2678, 3890, 2234, 4123, 3567])
n = len(AMS)

print("=" * 50)
print("PART A — AMS STATISTICS")
print("-" * 50)
print(f"Record length : {n:>3} years")
print(f"Mean          : {AMS.mean():.1f} m3/s")
print(f"Std deviation : {AMS.std():.1f} m3/s")

# CV: std/mean — dimensionless relative variability
# High CV = flashy river; Low CV = steady, regulated flow
print(f"CV            : {AMS.std()/AMS.mean():.3f}")
print(f"Min           : {AMS.min():.1f}")
print(f"Max           : {AMS.max():.1f}")
print(f"Median        : {np.median(AMS):.1f}")
print(f"Q25           : {np.percentile(AMS, 25):.1f}")
print(f"Q75           : {np.percentile(AMS, 75):.1f}")

# Skewness: third standardised moment — measures asymmetry
# Positive skew = long right tail (a few very large floods)
z    = (AMS - AMS.mean()) / AMS.std()
skew = (z**3).mean()
print(f"Skewness      : {skew:.3f}")

### 🔁 Try this

Compute the **Kurtosis** — the fourth standardised moment: `(z**4).mean()`.

Kurtosis > 3 indicates heavier-than-normal tails (more extreme events than a normal distribution would predict). What does the value tell you about this dataset?

---
## Code Block 2 — Part B: Weibull Flood Frequency Table

### What this code does

We sort the AMS in descending order, assign Weibull plotting positions, and build the full flood frequency table — then use `np.interp` to estimate design floods for standard return periods (T = 2, 5, 10, 25, 50 years).

### Why each step is taken

**Sort descending `np.sort(AMS)[::-1]`:**
Flood frequency analysis ranks floods from largest to smallest. Rank 1 = the largest observed flood. This gives the lowest exceedance probability (rarest event) to the highest flood.

**Weibull plotting position `P = rank/(n+1)`:**
The exceedance probability for rank i is `i/(n+1)`. This is the Weibull formula — it is unbiased and avoids P=0 or P=1. P is the probability of exceeding that flood in any given year.

**Return period `T = 1/P`:**
T is the average number of years between events of that magnitude or larger. T=10 means the flood is expected to be exceeded once in every 10 years — but this is an average, not a guarantee.

**`np.interp(1/T_target, P[::-1], AMS_s[::-1])` — design flood estimation:**
`np.interp` performs linear interpolation. We want the flood corresponding to `P = 1/T`. Since `np.interp` requires x to be increasing, we reverse both P and AMS_s so that P goes from small to large (using `[::-1]`).

### Algorithm

```
1. Sort AMS descending: AMS_s = np.sort(AMS)[::-1]

2. rank = np.arange(1, n+1)     → [1, 2, 3, ..., 25]
   P    = rank / (n+1)          → exceedance probabilities
   T    = 1 / P                 → return periods (years)

3. Print full table: Rank, Flow, P, T

4. For T_target in [2, 5, 10, 25, 50]:
   Q_T = np.interp(1/T_target, P[::-1], AMS_s[::-1])
   → linear interpolation to estimate design flood
```

### Expected output

```
==================================================
PART B — FLOOD FREQUENCY TABLE
--------------------------------------------------
 Rank   Flow(m3/s)   P(exceed)   T(years)
--------------------------------------------------
    1         5234       0.0385      26.00
    2         4890       0.0769      13.00
    3         4567       0.1154       8.67
  ...
   25         1234       0.9615       1.04

T=  2-yr flood: 2722 m3/s
T=  5-yr flood: 3948 m3/s
T= 10-yr flood: 4545 m3/s
T= 25-yr flood: 5002 m3/s
T= 50-yr flood: 5177 m3/s
```

In [ ]:
import numpy as np

AMS = np.array([1234, 2456, 1890, 3456, 2234, 4567, 3123, 1567, 2890, 5234,
                3456, 2123, 4567, 2890, 1678, 3234, 4890, 2345, 3678, 1456,
                2678, 3890, 2234, 4123, 3567])
n = len(AMS)

# Sort descending: largest flood gets rank 1
AMS_s = np.sort(AMS)[::-1]

# Weibull plotting positions
rank = np.arange(1, n + 1)
P    = rank / (n + 1)    # exceedance probability for each ranked event
T    = 1 / P             # return period (years)

print("=" * 55)
print("PART B — FLOOD FREQUENCY TABLE")
print("-" * 55)
print(f"{'Rank':>5} {'Flow(m3/s)':>12} {'P(exceed)':>12} {'T(years)':>10}")
print("-" * 55)
for i in range(n):
    print(f"{rank[i]:>5} {AMS_s[i]:>12.0f} {P[i]:>12.4f} {T[i]:>10.2f}")
print()

# Design flood estimation using linear interpolation
# np.interp(x, xp, fp): xp must be increasing → reverse P and AMS_s
for T_t in [2, 5, 10, 25, 50]:
    Q_T = np.interp(1/T_t, P[::-1], AMS_s[::-1])
    print(f"T={T_t:>3}-yr flood: {Q_T:.0f} m3/s")

### 🔁 Try this

Estimate the **T=100 year flood** using `np.interp`.

Note that 100 years > 26 years (our longest return period in the table). `np.interp` will extrapolate using the last two points — but this is statistically unreliable. What does this tell you about the limitations of the Weibull method for very long return periods?

---
## Code Block 3 — Part C: Flow Duration Curve from Daily Record

### What this code does

We generate a synthetic 122-day monsoon season record (July–October) and build the Flow Duration Curve. The FDC is the complement of the flood frequency table — it characterises the **full range** of flows, not just the annual peaks.

### Why each step is taken

**`np.concatenate([...])` with `np.random.exponential`:**
Each month has a different characteristic flow magnitude (scale parameter). Concatenating 4 monthly sub-arrays gives the full monsoon season record. The exponential distribution is a simple positive-valued distribution suitable for simulating daily streamflow.

**`np.interp(50, P, flow_s)` — Q50:**
Unlike `np.searchsorted` (used in Day 26), here we use `np.interp` directly. P is already sorted ascending (1.5%, 3%, ..., 98.5%), and flow_s is sorted descending. `np.interp` gives the exact interpolated value at P=50%, not just the nearest data point.

**Why `P` from 1 to n with step `/(n+1)`:**
The same Weibull formula as the flood frequency table. This ensures the exceedance probability is defined for every data point without reaching 0% or 100%.

**`Q10/Q90` — flashiness index:**
- Q10 = flow exceeded 10% of the time = a high-flow level
- Q90 = flow exceeded 90% of the time = near-drought level
- Ratio > 5 indicates a very flashy river; ratio < 2 indicates a stable, perennial river

### Algorithm

```
1. Generate 122-day synthetic monsoon record
   np.concatenate of 4 monthly sub-arrays

2. Sort descending: flow_s = np.sort(base_flow)[::-1]
   n = 122

3. Exceedance probability: P = arange(1,n+1)/(n+1)*100
   → sorted ascending (small P = rare high flow)

4. Q10 = np.interp(10, P, flow_s)
   Q50 = np.interp(50, P, flow_s)
   Q90 = np.interp(90, P, flow_s)

5. Flashiness = Q10/Q90

6. Plot: plt.semilogy(P, flow_s)
   Add horizontal lines at Q10, Q50, Q90
```

### Expected output

```
==================================================
PART C — FLOW DURATION CURVE
--------------------------------------------------
Total days    : 122
Mean flow     : 552.7 m3/s
Q10           : 1345.2 m3/s
Q50           : 412.8 m3/s
Q90           : 98.3 m3/s
Q10/Q90       : 13.69
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Synthetic monsoon season: Jul(30) + Aug(31) + Sep(30) + Oct(31) = 122 days
# Scale parameter = characteristic flow for each month
base_flow = np.concatenate([
    np.random.exponential(400, 30),   # July — rising limb
    np.random.exponential(800, 31),   # August — peak monsoon
    np.random.exponential(750, 30),   # September — sustained high
    np.random.exponential(500, 31),   # October — recession
])
base_flow = np.round(base_flow, 1)
n = len(base_flow)

# Sort descending for FDC
flow_s = np.sort(base_flow)[::-1]

# Weibull exceedance probability (%)
P = np.arange(1, n+1) / (n+1) * 100

# Interpolate to get exact percentile values
# np.interp(x, xp, fp) requires xp ascending — P is already ascending
Q10 = np.interp(10, P, flow_s)
Q50 = np.interp(50, P, flow_s)
Q90 = np.interp(90, P, flow_s)

print("=" * 50)
print("PART C — FLOW DURATION CURVE")
print("-" * 50)
print(f"Total days    : {n}")
print(f"Mean flow     : {base_flow.mean():.1f} m3/s")
print(f"Q10           : {Q10:.1f} m3/s")
print(f"Q50           : {Q50:.1f} m3/s")
print(f"Q90           : {Q90:.1f} m3/s")
print(f"Q10/Q90       : {Q10/Q90:.2f}")

plt.figure(figsize=(8, 5))
plt.semilogy(P, flow_s, 'b-', linewidth=1.5)
for Q, lbl, col in [(Q10,'Q10','orange'), (Q50,'Q50','green'), (Q90,'Q90','red')]:
    plt.axhline(Q, color=col, linestyle='--', label=f'{lbl}={Q:.0f}')
plt.xlabel('Exceedance Probability (%)')
plt.ylabel('Discharge (m3/s) — log scale')
plt.title('Flow Duration Curve — Monsoon Season')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 🔁 Try this

Compute Q25 (flow exceeded 25% of the time) and Q75 (exceeded 75% of the time).

- Which represents the "high flow season" threshold?
- Add them as additional lines to the FDC plot.

---
## Unit 3 — Complete Summary of NumPy Concepts

| Day | Key concept | Core function |
|---|---|---|
| 19 | ndarray basics, statistics | `np.array`, `.sum()`, `.mean()`, boolean indexing |
| 20 | 1-D, 2-D, 3-D arrays | axis=0/1, `np.linalg.solve` |
| 21 | Indexing and slicing | `arr[i:j]`, `arr[:,c]`, `arr[[i,j,k]]` |
| 22 | Cumulative ops, rolling mean | `np.cumsum`, `np.convolve`, `np.diff` |
| 23 | Boolean arrays, QC | `np.select`, `np.bincount`, `np.nanmean` |
| 24 | Broadcasting, vectorised formulas | `reshape(-1,1)`, SCS-CN vectorised, pipe table |
| 25 | Reshape, transpose, stack | `.reshape`, `.T`, `np.vstack`, z-score |
| 26 | File I/O, FDC | `np.savetxt`, `np.loadtxt`, `np.searchsorted` |
| 27 | Flood frequency analysis | `np.sort`, Weibull, `np.interp`, FDC |

---
## Day 27 — Unit 3 Capstone Assignment

Submit your own complete flood analysis notebook.

Using the 25-year AMS from this session:

1. Compute and print all statistics from Part A
2. Build the full Weibull flood frequency table (Part B)
3. Estimate the T = 2, 5, 10, 25, 50 year design floods using `np.interp`
4. Plot the flood frequency curve: log(T) on x-axis vs Flow on y-axis (use `plt.semilogx`)
5. Write one paragraph interpreting the results for a dam safety engineer

**Submit your completed notebook to GitHub with commit message:**
`Day 27 — Unit 3 capstone completed`

### ▶ Capstone cell

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

AMS = np.array([1234, 2456, 1890, 3456, 2234, 4567, 3123, 1567, 2890, 5234,
                3456, 2123, 4567, 2890, 1678, 3234, 4890, 2345, 3678, 1456,
                2678, 3890, 2234, 4123, 3567])

# 1. Statistics
n = AMS.size
print(f"Mean: {AMS.mean():.1f}  Std: {AMS.std():.1f}  CV: {AMS.std()/AMS.mean():.3f}")

# 2. Weibull table
AMS_s = np.sort(AMS)[::-1]
rank  = np.arange(1, n+1)
P     = rank / (n+1)
T     = 1 / P
print(f"
{'Rank':>5} {'Flow':>10} {'P':>10} {'T':>10}")
for i in range(n):
    print(f"{rank[i]:>5} {AMS_s[i]:>10.0f} {P[i]:>10.4f} {T[i]:>10.2f}")

# 3. Design floods
print()
for T_t in [2, 5, 10, 25, 50]:
    Q_T = np.interp(1/T_t, P[::-1], AMS_s[::-1])
    print(f"T={T_t:>3} yr: {Q_T:.0f} m3/s")

# 4. Flood frequency plot
plt.figure(figsize=(8,5))
plt.semilogx(T, AMS_s, 'bo-', markersize=4, label='Observed peaks')
plt.xlabel('Return Period T (years) — log scale')
plt.ylabel('Peak Discharge (m3/s)')
plt.title('Flood Frequency Curve — Weibull (Cauvery at KRS)')
plt.grid(which='both', alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

# 5. Your interpretation paragraph here:
print("\nInterpretation:")
print("The 25-year record shows a mean annual flood of ... m3/s.")
print("The 10-year design flood of ... m3/s would be used for ...")

---
- [ ] Run all cells — verify outputs match expected outputs in each block
- [ ] Complete the capstone cell — replace print statements with your interpretation
- [ ] Upload to GitHub: `Unit3_NumPy/CE541E08_U3_Day27.ipynb`
- [ ] Commit message: `Day 27 — Unit 3 capstone completed`

**Unit 4 starts next: The Pandas Library**

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*